# CT4110 – Topic 6 Assignment
## Quantum Fourier Transform and Quantum Phase Estimation

In this exercise, you will use Qiskit to construct the Quantum Fourier Transform (QFT), compare a manual circuit with Qiskit's `QFTGate`, verify the inverse transform, and use inverse QFT inside Quantum Phase Estimation (QPE).

For each question, run every checkpoint in order and include the requested circuit diagrams, numerical results, and short explanations. Complete all `YOUR CODE HERE` and `ANSWER: [ ]` sections.

Each checkpoint ends with a **REQUIRED OUTPUT CHECK**. Before submission, make sure the listed output is visible below the cell.

In [ ]:
# ============================================================
# Course Setup – Run This Cell First
# ============================================================

# Install the packages only when they are not already available.
try:
    import qiskit
    import qiskit_aer
    import pylatexenc
except ImportError:
    %pip install -q qiskit qiskit-aer matplotlib pylatexenc

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

print("Qiskit setup complete.")

# REQUIRED OUTPUT CHECK
# The console must display: Qiskit setup complete.

## Question 1 – Construct the Lecture QFT Circuit

For a vector $(x_0,x_1,\ldots,x_{N-1})$, use the lecture convention

$$y_j=\sum_{k=0}^{N-1}x_k e^{-i2\pi jk/N},\qquad j=0,1,\ldots,N-1.$$

For a basis state $|j\rangle$, the QFT used in this exercise is

$$\operatorname{QFT}|j\rangle=\frac{1}{\sqrt{N}}\sum_{k=0}^{N-1}e^{i2\pi jk/N}|k\rangle.$$

The opposite signs are the convention used in the lecture notes. The factor $1/\sqrt{N}$ normalises the quantum state. For three qubits, $N=2^3=8$.

Use the nontrivial input $|001\rangle=|1\rangle$. Follow the lecture labels $j_1=q_0$, $j_2=q_1$, and $j_3=q_2$. Prepare the input by applying $X$ to `q2`.

Build the lecture's three-qubit QFT circuit in this order:

1. apply $H$ to `q0`, then controlled phases $R_2$ from `q1` to `q0` and $R_3$ from `q2` to `q0`;
2. apply $H$ to `q1`, then a controlled phase $R_2$ from `q2` to `q1`;
3. apply $H$ to `q2`;
4. swap `q0` and `q2`.

### Controlled-phase rotation in Qiskit

A controlled-phase rotation needs three pieces of information: the rotation angle, the control qubit, and the target qubit. Qiskit uses

```python
circuit.cp(angle, control_qubit, target_qubit)
```

For the lecture gate $R_k$, calculate the angle as $2\pi/2^k$. For example, $R_2$ uses $\pi/2$ and $R_3$ uses $\pi/4$. The phase is applied when both participating qubits are in state $|1\rangle$. Choose the control and target indices from the circuit sequence given above.

Qiskit reference: [`QuantumCircuit.cp()`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.QuantumCircuit#cp).

In [ ]:
# Question 1 — Checkpoint 1: Build the lecture QFT
#
# PHASE 1 – Define the rotation angles
# Define a function named theta that accepts k and returns 2*pi / 2^k.
#
# PHASE 2 – Prepare the input
# Create a three-qubit circuit named manual_qft.
# Prepare |001> using the lecture's wire labels and add a labelled barrier.
#
# PHASE 3 – Build the QFT
# Add the Hadamard, controlled-phase and SWAP gates in the order stated
# in Question 1. Use theta(2) for R2 and theta(3) for R3.
#
# PHASE 4 – Display the circuit
# Draw manual_qft using the Matplotlib circuit drawer.
#
# YOUR CODE HERE
#
# REQUIRED OUTPUT CHECK
# The diagram must contain 3 H gates, 3 controlled-phase gates and 1 SWAP.

### Convert to Qiskit's displayed bit order

A QFT encodes information in relative phases. Measuring immediately would hide those phases, so use the exact `Statevector(circuit)` method introduced in Topic 4.

The lecture circuit labels the top wire as the first binary digit, whereas Qiskit treats `q0` as the least-significant qubit and prints measured bit strings with the highest-index bit on the left. Calling `reverse_bits()` converts the lecture-labelled circuit to Qiskit's convention, allowing its statevector and measurement results to be compared directly with `QFTGate`.

Qiskit references: [bit ordering and `reverse_bits()`](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering) and [`Statevector`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector).

In [ ]:
# Question 1 — Checkpoint 2: Match Qiskit's setting and inspect the statevector
#
# PHASE 1 – Match Qiskit's bit setting
# Use reverse_bits() on manual_qft and store the resulting circuit as
# qft_qiskit_setting.
#
# PHASE 2 – Inspect the state
# Create manual_state from qft_qiskit_setting using the Topic 4
# Statevector(circuit) method.
# Print all amplitudes rounded to three decimal places.
#
# YOUR CODE HERE
#
# REQUIRED OUTPUT CHECK
# The output must be a statevector containing 8 complex amplitudes.


# Question 1 — Checkpoint 3: Measure using the Qiskit setting
#
# PHASE 1 – Prepare a measured circuit
# Copy qft_qiskit_setting into measured_qft and measure every qubit.
#
# PHASE 2 – Simulate
# Create an AerSimulator, transpile measured_qft for it and run 1,000 shots.
# Store the resulting counts as qft_counts.
#
# PHASE 3 – Display results
# Print qft_counts and display a histogram.
#
# YOUR CODE HERE
#
# REQUIRED OUTPUT CHECK
# All eight 3-bit outcomes should appear with approximately equal counts.

## Question 2 – Recreate the QFT with Qiskit's `QFTGate`

Recreate the same transformation using Qiskit's `QFTGate` class. Because the circuit from Question 1 was adjusted to Qiskit's setting before inspection, prepare the matching input $|001\rangle$ here by applying $X$ to `q0`, then append `QFTGate(3)` to `[0, 1, 2]`.

Compare the Question 2 statevector with `manual_state` from Question 1 using `Statevector(...).equiv(...)`. This checks equality up to an unobservable global phase and confirms that the manual lecture circuit matches Qiskit's gate implementation.

Qiskit references: [`QFTGate`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.QFTGate) and [`Statevector.equiv()`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector#equiv).

In [ ]:
# Question 2 — Checkpoint 1: Build the Qiskit QFT circuit
#
# PHASE 1 – Prepare the input
# Create a three-qubit circuit named qft_library and prepare |001> using
# Qiskit's qubit convention.
#
# PHASE 2 – Add Qiskit's QFT
# Append a three-qubit QFTGate to all three qubits.
#
# PHASE 3 – Display the implementation
# Decompose the circuit and draw it using the Matplotlib circuit drawer.
#
# YOUR CODE HERE
#
# REQUIRED OUTPUT CHECK
# The decomposed diagram must show a complete three-qubit QFT after the X gate.

In [ ]:
# Question 2 — Checkpoint 2: Compare the two QFT constructions
#
# PHASE 1 – Obtain the library state
# Create library_state from qft_library using Statevector(circuit).
#
# PHASE 2 – Compare
# Use the Statevector equivalence method to compare manual_state with
# library_state and print the result with a clear label.
#
# PHASE 3 – Interpret
# Explain what a result of True demonstrates.
#
# YOUR CODE HERE
# ANSWER: [ ]
#
# REQUIRED OUTPUT CHECK
# The comparison must print: Manual circuit matches QFTGate: True

## Question 3 – Reverse the QFT with the Inverse QFT

The inverse QFT is $\operatorname{QFT}^{\dagger}$. The lecture notes use the identity

$$\operatorname{QFT}^{\dagger}\operatorname{QFT}|\psi\rangle=|\psi\rangle.$$

Start from `qft_qiskit_setting`, the manual QFT circuit prepared for Qiskit's setting in Question 1. Append `QFTGate(3).inverse()` to that circuit to reverse the manually constructed QFT. Do **not** start from the `QFTGate` circuit created in Question 2.

First verify the recovery of $|001\rangle$ exactly with a statevector. Then copy the circuit, measure **all three qubits**, and run it on the simulator. Because the round trip is deterministic, the expected result is `001` on every ideal shot; no statistical search for a “most frequent” state is needed.

Qiskit reference: [`Instruction.inverse()`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.Instruction#inverse).

In [ ]:
# Question 3 — Checkpoint 1: Append inverse QFT to the Question 1 circuit
#
# PHASE 1 – Reuse Question 1
# Copy qft_qiskit_setting into a new circuit named roundtrip_qc.
# Do not use qft_library from Question 2.
#
# PHASE 2 – Reverse the transform
# Append the inverse of a three-qubit QFTGate to roundtrip_qc.
# Do not add measurements yet.
#
# PHASE 3 – Verify with statevectors
# Create roundtrip_state from the circuit and a target_state for |001>.
# Test their equivalence and print the result with a clear label.
#
# PHASE 4 – Display the circuit
# Draw roundtrip_qc using the Matplotlib circuit drawer.
#
# YOUR CODE HERE
#
# REQUIRED OUTPUT CHECK
# The statevector comparison must print: Round trip returns |001>: True

In [ ]:
# Question 3 — Checkpoint 2: Measure the recovered state
#
# PHASE 1 – Add measurement
# Copy roundtrip_qc into measured_roundtrip and measure every qubit.
#
# PHASE 2 – Simulate
# Run measured_roundtrip on AerSimulator for 1,000 shots and store the
# resulting counts in counts.
#
# PHASE 3 – Display and interpret
# Print counts, plot a histogram and explain why the ideal result is
# deterministic.
#
# YOUR CODE HERE
# ANSWER: [ ]
#
# REQUIRED OUTPUT CHECK
# The ideal result must be {'001': 1000}.

## Question 4 – Hard Task: Quantum Phase Estimation

Quantum Phase Estimation estimates $\phi$ when an eigenstate satisfies

$$U|u\rangle=e^{i2\pi\phi}|u\rangle.$$

Use three counting qubits and one eigenstate qubit to estimate an unknown eigenphase $\phi$. The supplied unitary is a phase rotation through

$$\alpha=\frac{\pi}{4}.$$

Prepare the eigenstate qubit in $|1\rangle$. Do not calculate or hard-code $\phi$ before running QPE. Instead, construct the controlled powers of the supplied unitary and infer $\phi$ from the measured counting register. Pair the power $2^j$ with counting qubit `q[n_counting - 1 - j]`; the corresponding controlled rotation angle is

$$\alpha\,2^j.$$

### Controlled powers for QPE

For each $j=0,1,2$, calculate the powered rotation angle as `alpha * (2 ** j)`. Apply that angle with

```python
qpe_qc.cp(powered_angle, control_qubit, eigenstate_qubit)
```

Use `control_qubit = n_counting - 1 - j`. This pairs $U$, $U^2$, and $U^4$ with the counting qubits in the order required by the bit convention used in this exercise. Repeat the controlled-phase operation once for each value of $j$.

Apply inverse QFT to the counting register in the order `[2, 1, 0]`, measure every counting qubit into the classical bit with the same index, and then use `reverse_bits()` for the final displayed circuit. Derive the phase estimate only from the simulator output.

Qiskit reference: [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html).

In [ ]:
# Question 4 — Checkpoint 1: Build and draw the QPE circuit
#
# PHASE 1 – Define the problem
# Set n_counting to 3, identify the eigenstate qubit, store the supplied
# rotation angle alpha, and create qpe_qc with four qubits and three
# classical bits. Do not define the unknown eigenphase.
#
# PHASE 2 – Prepare the registers
# Prepare the eigenstate qubit in |1> and put all counting qubits into
# superposition. Add a barrier after preparation.
#
# PHASE 3 – Encode the phase
# Apply the three controlled powers described in Question 4. Pair each
# power 2^j with counting qubit n_counting - 1 - j. Add a barrier.
#
# PHASE 4 – Decode the phase
# Append inverse QFT to the counting qubits in the order [2, 1, 0].
# Add a barrier after inverse QFT.
#
# PHASE 5 – Measure and display
# Measure each counting qubit into the classical bit with the same index.
# Use reverse_bits() after measurement, then draw the complete circuit.
#
# YOUR CODE HERE
#
# REQUIRED OUTPUT CHECK
# The diagram must show 3 H gates, 3 controlled-phase gates, one inverse-QFT
# block, and measurements on all 3 counting qubits.

In [ ]:
# Question 4 — Checkpoint 2: Run QPE and calculate the phase
#
# PHASE 1 – Execute
# Run qpe_qc on AerSimulator for 1,000 shots and store the counts as
# qpe_counts. Display a histogram.
#
# PHASE 2 – Identify the measured phase
# Find the most likely bit string and convert it from binary to an integer.
#
# PHASE 3 – Calculate
# Divide the measured integer by 2^n_counting to obtain estimated_phi.
#
# PHASE 4 – Report
# Print the counts, most likely bit string, measured integer and estimated
# phase with clear labels.
#
# YOUR CODE HERE
#
# REQUIRED OUTPUT CHECK
# The output must show a dominant 3-bit result and a phase estimate derived
# from that result. The phase must not be hard-coded.

In [ ]:
# Question 4 — Checkpoint 3: Show the phase calculation
#
# PHASE 1 – Binary conversion
# Write the conversion from the most likely binary string to integer x.
# ANSWER: [ ]
#
# PHASE 2 – Phase estimate
# Show phi_estimated = x / 2^n for n = 3.
# ANSWER: [ ]
#
# REQUIRED OUTPUT CHECK
# Your written work must show the measured binary string, its integer value
# and the phase obtained by dividing that integer by 2^3.